# XAI on LARS detectors

Three detectors are compared with two families of XAI methods.

**Model-agnostic (same recipe for all three):**
- **D-RISE** — black-box saliency via random masking + detection-similarity scoring.

**Model-specific:**
- **Faster R-CNN:** Grad-CAM++ on the FPN backbone.
- **YOLOv8:**     EigenCAM on the head's last C2f activation.
- **RF-DETR:**    Cross-attention map from the last decoder layer (deformable attention scattered onto the image grid).

All cells use the shared setup (sample image, models, helpers) from the first cell.

In [1]:
# ── Shared setup ─────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import random

SEED = 4
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMAGE_DIR  = Path('../Data/lars_processed/test/images')

# Get all jpg files
images = list(IMAGE_DIR.glob("*.jpg"))

# Pick one randomly
# SAMPLE_IMG = random.choice(images)
SAMPLE_IMG = IMAGE_DIR / 'yt015_03_00049.jpg'


# Model weights are symlinked in ../model_showcase/
WEIGHTS    = Path('../model_showcase')
FASTERRCNN = WEIGHTS / 'fasterrcnn_trial002.pth'
RFDETR     = WEIGHTS / 'rfdetr_trial004.pth'
YOLO_PTH   = WEIGHTS / 'yolo_exp7.pt'

N_CLASSES = 8
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONF_FASTERRCNN = 0.7
CONF_RFDETR     = 0.4
CONF_YOLO       = 0.3

# Load models ----------------------------------------------------------------
sys.path.insert(0, str(Path('../3_Model').resolve()))
import torchvision.transforms.functional as TF
from train_fasterrcnn import build_model as build_frcnn

frcnn = build_frcnn('base', num_classes=N_CLASSES + 1)
frcnn.load_state_dict(torch.load(FASTERRCNN, map_location=DEVICE))
frcnn.to(DEVICE).eval();

from rfdetr import RFDETRBase
rfdetr = RFDETRBase(resolution=784, num_classes=N_CLASSES,
                    pretrain_weights=str(RFDETR))
# NOTE: don't call optimize_for_inference() — we need the raw torch model
# (with grad / hookable submodules) for the model-specific XAI cells.

from ultralytics import YOLO as _YOLO
yolo = _YOLO(str(YOLO_PTH))

# Sample image ---------------------------------------------------------------
pil_img    = Image.open(SAMPLE_IMG).convert('RGB')
img_np     = np.array(pil_img)
H_IMG, W_IMG = img_np.shape[:2]
print(f'device={DEVICE}  image={SAMPLE_IMG.name}  size={W_IMG}x{H_IMG}')

# Visualisation helpers ------------------------------------------------------
def overlay_heatmap(img_rgb, sal, alpha=0.45, cmap=cv2.COLORMAP_JET):
    """img_rgb: HxWx3 uint8; sal: HxW float in [0,1] → HxWx3 uint8 overlay."""
    sal = np.clip(sal, 0, 1)
    sal_u8 = (sal * 255).astype(np.uint8)
    if sal_u8.shape[:2] != img_rgb.shape[:2]:
        sal_u8 = cv2.resize(sal_u8, (img_rgb.shape[1], img_rgb.shape[0]))
    heat = cv2.applyColorMap(sal_u8, cmap)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(heat, alpha, img_rgb, 1 - alpha, 0)

def draw_box(ax, box, color='lime', label=None):
    x1, y1, x2, y2 = box
    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                               fill=False, edgecolor=color, lw=2))
    if label:
        ax.text(x1, max(0, y1-4), label, color=color, fontsize=9,
                bbox=dict(facecolor='black', alpha=0.5, pad=1, lw=0))

def show(panels, titles, figsize=(18, 6)):
    fig, axes = plt.subplots(1, len(panels), figsize=figsize)
    if len(panels) == 1: axes = [axes]
    for ax, p, t in zip(axes, panels, titles):
        ax.imshow(p); ax.set_title(t); ax.axis('off')
    plt.tight_layout(); plt.show()


device=cuda  image=yt015_03_00049.jpg  size=1280x720


In [ ]:
# ── Save XAI figures into the XAI folder ─────────────────────────────────────
# Every plotting cell below calls save_fig(fig, '<name>', **params) just before
# plt.show(). Figures are always written as PNGs into ./plots/ (inside this
# 5_XAI folder). The sample image stem and every parameter passed are encoded
# into the filename, so the model and the settings that produced a plot can be
# read straight off its name, e.g.
#   xai_drise_fasterrcnn__img-yt015_03_00049__n_masks-3000__s-20__p-0.1.png
XAI_PLOT_DIR = Path('plots').resolve()
XAI_PLOT_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, **params):
    """Save `fig` to ./plots/<name>[__key-val...].png inside the XAI folder.

    The sample image stem and every keyword in `params` are appended to the
    filename so the model and parameters used to generate the plot are captured
    in its name (dpi 130)."""
    parts = [name, f'img-{SAMPLE_IMG.stem}']            # always tag the image
    parts += [f'{k}-{v}' for k, v in params.items()]    # encode each parameter
    out = XAI_PLOT_DIR / ('__'.join(parts) + '.png')
    fig.savefig(out, dpi=130, bbox_inches='tight')
    print(f'saved -> {out}')

print(f'XAI figures will be saved to {XAI_PLOT_DIR}')

images: image=yt015_03_00049.jpg

In [ ]:
# ── Sanity check: detections from each model on the sample image ────────────
# (Run after cell-drise-impl so the _*_predict helpers exist.)
# If you haven't run that cell yet, uncomment the inline predictors below.

def _predict_all():
    with torch.no_grad():
        out = frcnn([TF.to_tensor(img_np).to(DEVICE)])[0]
    fr_keep = (out['scores'].cpu().numpy() >= CONF_FASTERRCNN)
    fr = (out['boxes'].cpu().numpy()[fr_keep],
          out['labels'].cpu().numpy()[fr_keep] - 1,   # 0-indexed
          out['scores'].cpu().numpy()[fr_keep])

    rd = rfdetr.predict(pil_img, threshold=CONF_RFDETR)
    rd = (np.asarray(rd.xyxy), np.asarray(rd.class_id), np.asarray(rd.confidence))

    yr = yolo.predict(source=pil_img, imgsz=1024, conf=CONF_YOLO,
                      device=DEVICE, verbose=False)[0]
    yo = (yr.boxes.xyxy.cpu().numpy(),
          yr.boxes.cls.cpu().numpy().astype(int),
          yr.boxes.conf.cpu().numpy())
    return fr, rd, yo

fr, rd, yo = _predict_all()
print(f'Faster R-CNN @ conf≥{CONF_FASTERRCNN}: {len(fr[0])} detections')
print(f'RF-DETR      @ conf≥{CONF_RFDETR}:    {len(rd[0])} detections')
print(f'YOLO         @ conf≥{CONF_YOLO}:    {len(yo[0])} detections')

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, (boxes, cls, scr), title in zip(
    axes,
    [fr, rd, yo],
    [f'Faster R-CNN ({len(fr[0])})',
     f'RF-DETR ({len(rd[0])})',
     f'YOLO ({len(yo[0])})'],
):
    ax.imshow(img_np); ax.set_title(title); ax.axis('off')
    for b, c, s in zip(boxes, cls, scr):
        draw_box(ax, b, color='lime', label=f'{int(c)}:{s:.2f}')
plt.tight_layout()
# Filename records the confidence threshold used for each model's detections.
save_fig(fig, 'xai_detections_all_models',
         frcnn_conf=CONF_FASTERRCNN, rfdetr_conf=CONF_RFDETR, yolo_conf=CONF_YOLO)
plt.show()

## 1 — D-RISE (model-agnostic)

Black-box saliency for object detection (Petsiuk et al., 2021). The idea:

1. Sample many low-resolution binary masks, upsample them, multiply pixel-wise into the input image.
2. Run the detector on each masked image and find the prediction most similar to a chosen *target detection* from the unmasked image.
3. Score each mask by that similarity (IoU × class-match × confidence).
4. Saliency = weighted average of the masks.

The same procedure is used for all three detectors — only the `predict_fn` adapter changes.

In [4]:
# ── D-RISE core ─────────────────────────────────────────────────────────────
from typing import Callable, List, Tuple

def _iou(box, boxes):
    if len(boxes) == 0: return np.zeros(0)
    x1 = np.maximum(box[0], boxes[:,0]); y1 = np.maximum(box[1], boxes[:,1])
    x2 = np.minimum(box[2], boxes[:,2]); y2 = np.minimum(box[3], boxes[:,3])
    inter = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    a1 = (box[2]-box[0]) * (box[3]-box[1])
    a2 = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
    return inter / np.maximum(a1 + a2 - inter, 1e-9)

def generate_masks(n_masks, s, p, H, W, rng):
    """RISE masks: sxs Bernoulli grid, bilinear-upsampled, random crop to HxW."""
    cell_h, cell_w = int(np.ceil(H / s)), int(np.ceil(W / s))
    up_h, up_w = (s + 1) * cell_h, (s + 1) * cell_w
    grid = (rng.random((n_masks, s, s)) < p).astype(np.float32)
    masks = np.empty((n_masks, H, W), dtype=np.float32)
    for i in range(n_masks):
        big = cv2.resize(grid[i], (up_w, up_h), interpolation=cv2.INTER_LINEAR)
        oy = rng.integers(0, cell_h + 1)
        ox = rng.integers(0, cell_w + 1)
        masks[i] = big[oy:oy + H, ox:ox + W]
    return masks

def drise(
    img_rgb: np.ndarray,
    target_box: np.ndarray,        # xyxy in image coords
    target_class: int,
    predict_fn: Callable[[np.ndarray], Tuple[np.ndarray, np.ndarray, np.ndarray]],
    n_masks: int = 5000,
    s: int = 6,
    p: float = 0.1,
    batch: int = 8,
    seed: int = 4,
):
    """Returns HxW saliency map in [0,1].

    predict_fn(masked_img_rgb_uint8) → (boxes_xyxy[N,4], class_ids[N], scores[N]).
    Similarity for a mask = max over preds of  IoU(target, pred) · 1[class==] · score.
    """
    rng = np.random.default_rng(seed)
    H, W = img_rgb.shape[:2]
    masks = generate_masks(n_masks, s, p, H, W, rng)

    weights = np.zeros(n_masks, dtype=np.float32)
    for i in range(0, n_masks, batch):
        chunk = masks[i:i+batch]
        for j, m in enumerate(chunk):
            masked = (img_rgb.astype(np.float32) * m[..., None]).astype(np.uint8)
            boxes, cls, scr = predict_fn(masked)
            if len(boxes) == 0:
                weights[i+j] = 0.0
                continue
            ious   = _iou(target_box, boxes)
            match  = (cls == target_class).astype(np.float32)
            sims   = ious * match * scr
            weights[i+j] = sims.max()

    if weights.sum() <= 0:
        return np.zeros((H, W), dtype=np.float32)
    sal = (weights[:, None, None] * masks).sum(0) / weights.sum()
    sal -= sal.min(); sal /= max(sal.max(), 1e-9)
    return sal

# ── Per-model predict adapters ──────────────────────────────────────────────
def _frcnn_predict(img_rgb_u8):
    with torch.no_grad():
        t = TF.to_tensor(img_rgb_u8).to(DEVICE)
        out = frcnn([t])[0]
    keep = out['scores'].cpu() >= 0.05
    return (out['boxes'][keep].cpu().numpy(),
            out['labels'][keep].cpu().numpy() - 1,   # back to 0-indexed
            out['scores'][keep].cpu().numpy())

def _rfdetr_predict(img_rgb_u8):
    pil = Image.fromarray(img_rgb_u8)
    dets = rfdetr.predict(pil, threshold=0.05)
    if len(dets) == 0:
        return np.zeros((0,4)), np.zeros(0,int), np.zeros(0)
    return np.asarray(dets.xyxy), np.asarray(dets.class_id), np.asarray(dets.confidence)

def _yolo_predict(img_rgb_u8):
    res = yolo.predict(source=Image.fromarray(img_rgb_u8),
                       imgsz=1024, conf=0.05,
                       device=DEVICE, verbose=False)[0]
    if len(res.boxes) == 0:
        return np.zeros((0,4)), np.zeros(0,int), np.zeros(0)
    return (res.boxes.xyxy.cpu().numpy(),
            res.boxes.cls.cpu().numpy().astype(int),
            res.boxes.conf.cpu().numpy())

print('D-RISE ready. Tunables: n_masks, s (grid), p (keep prob).')

D-RISE ready. Tunables: n_masks, s (grid), p (keep prob).


In [ ]:
# ── D-RISE: Faster R-CNN ────────────────────────────────────────────────────
# Pick the highest-confidence detection as the explanation target.
boxes, cls, scr = _frcnn_predict(img_np)
keep = scr >= CONF_FASTERRCNN
boxes, cls, scr = boxes[keep], cls[keep], scr[keep]
idx = int(scr.argmax())
tgt_box, tgt_cls = boxes[idx], int(cls[idx])
print(f'Target detection: class={tgt_cls}  score={scr[idx]:.3f}  box={tgt_box.round(1)}')

# D-RISE hyperparameters (also encoded into the saved filename below)
N_MASKS, GRID_S, KEEP_P = 3000, 20, 0.10
sal = drise(img_np, tgt_box, tgt_cls, _frcnn_predict,
            n_masks=N_MASKS, s=GRID_S, p=KEEP_P, batch=1, seed=SEED)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(img_np); ax[0].set_title('Input'); ax[0].axis('off')
draw_box(ax[0], tgt_box, label=f'frcnn cls={tgt_cls} s={scr[idx]:.2f}')
ax[1].imshow(overlay_heatmap(img_np, sal)); ax[1].set_title('D-RISE — Faster R-CNN')
draw_box(ax[1], tgt_box)
ax[1].axis('off'); plt.tight_layout()
save_fig(fig, 'xai_drise_fasterrcnn',
         n_masks=N_MASKS, s=GRID_S, p=KEEP_P, conf=CONF_FASTERRCNN)
plt.show()

In [ ]:
# ── D-RISE: RF-DETR ─────────────────────────────────────────────────────────
boxes, cls, scr = _rfdetr_predict(img_np)
keep = scr >= CONF_RFDETR
boxes, cls, scr = boxes[keep], cls[keep], scr[keep]
idx = int(scr.argmax())
tgt_box, tgt_cls = boxes[idx], int(cls[idx])
print(f'Target detection: class={tgt_cls}  score={scr[idx]:.3f}  box={tgt_box.round(1)}')

# D-RISE hyperparameters (also encoded into the saved filename below)
N_MASKS, GRID_S, KEEP_P = 3000, 20, 0.10
sal = drise(img_np, tgt_box, tgt_cls, _frcnn_predict,
            n_masks=N_MASKS, s=GRID_S, p=KEEP_P, batch=1, seed=SEED)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(img_np); ax[0].set_title('Input'); ax[0].axis('off')
draw_box(ax[0], tgt_box, label=f'rfdetr cls={tgt_cls} s={scr[idx]:.2f}')
ax[1].imshow(overlay_heatmap(img_np, sal)); ax[1].set_title('D-RISE — RF-DETR')
draw_box(ax[1], tgt_box)
ax[1].axis('off'); plt.tight_layout()
save_fig(fig, 'xai_drise_rfdetr',
         n_masks=N_MASKS, s=GRID_S, p=KEEP_P, conf=CONF_RFDETR)
plt.show()

In [ ]:
# ── D-RISE: YOLO ────────────────────────────────────────────────────────────
boxes, cls, scr = _yolo_predict(img_np)
keep = scr >= CONF_YOLO
boxes, cls, scr = boxes[keep], cls[keep], scr[keep]
idx = int(scr.argmax())
tgt_box, tgt_cls = boxes[idx], int(cls[idx])
print(f'Target detection: class={tgt_cls}  score={scr[idx]:.3f}  box={tgt_box.round(1)}')

# D-RISE hyperparameters (also encoded into the saved filename below)
N_MASKS, GRID_S, KEEP_P = 3000, 20, 0.10
sal = drise(img_np, tgt_box, tgt_cls, _frcnn_predict,
            n_masks=N_MASKS, s=GRID_S, p=KEEP_P, batch=1, seed=SEED)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(img_np); ax[0].set_title('Input'); ax[0].axis('off')
draw_box(ax[0], tgt_box, label=f'yolo cls={tgt_cls} s={scr[idx]:.2f}')
ax[1].imshow(overlay_heatmap(img_np, sal)); ax[1].set_title('D-RISE — YOLO')
draw_box(ax[1], tgt_box)
ax[1].axis('off'); plt.tight_layout()
save_fig(fig, 'xai_drise_yolo',
         n_masks=N_MASKS, s=GRID_S, p=KEEP_P, conf=CONF_YOLO)
plt.show()

## 2 — Grad-CAM++ (Faster R-CNN, RoI-head)

White-box, gradient-based saliency. We apply Grad-CAM++ **after `roi_align`** —
hooking the last conv stage of `box_head` (whose output is `[N, 256, 7, 7]`) —
instead of the FPN. This makes the explanation per-detection and naturally
aligned with the predicted box.

For the chosen detection we backprop the classifier logit for the predicted
class to that 7×7 feature map and apply

$$\alpha_k^{c} = \frac{(\partial Y^c/\partial A^k)^2}{2(\partial Y^c/\partial A^k)^2 + \sum_{ij} A^k_{ij} (\partial Y^c/\partial A^k)^3}$$

The resulting 7×7 saliency is bicubically resized to the box region and
pasted into an image-shaped canvas (zero outside the box).

In [ ]:
# ── Grad-CAM++ for Faster R-CNN (RoI-head version) ─────────────────────────
# Hooks the LAST conv layer of box_head (box_head[3]) instead of the FPN.
# After roi_align the features are [N, 256, 7, 7] and live in the cropped box
# coordinate system, so the resulting CAM is naturally per-detection and aligned
# with the predicted box. We resize the 7×7 map back to the box region in the
# original image, leaving the area outside the box at zero.

from torchvision.ops import roi_align

frcnn.eval()
# Keep parameter requires_grad=True so the activation tensor stays in the
# autograd graph (we won't actually use the param grads).

img_t = TF.to_tensor(img_np).to(DEVICE).unsqueeze(0)

# 1) pick the target detection (highest-confidence above threshold)
with torch.no_grad():
    out = frcnn(img_t)[0]
keep = (out['scores'] >= CONF_FASTERRCNN).cpu().numpy()
all_boxes = out['boxes'].cpu().numpy(); all_scr = out['scores'].cpu().numpy()
all_lbl = out['labels'].cpu().numpy()
idx = int(np.argmax(all_scr * keep))
tgt_box = all_boxes[idx]; tgt_lbl = int(all_lbl[idx])
print(f'Target: label={tgt_lbl}  score={all_scr[idx]:.3f}  box={tgt_box.round(1)}')

# 2) hook the last conv stage of the box_head — its output is [1, 256, 7, 7]
act_store = {}
def _capture(module, inp, out):
    out.retain_grad()
    act_store['feat'] = out
h = frcnn.roi_heads.box_head[3].register_forward_hook(_capture)

# 3) re-run backbone, roi_align the target box, push through box_head/predictor
fpn = frcnn.backbone(img_t)
img_shapes = [(img_np.shape[0], img_np.shape[1])]
roi = torch.tensor([tgt_box.tolist()], device=DEVICE, dtype=torch.float32)
box_feats = frcnn.roi_heads.box_roi_pool(fpn, [roi], img_shapes)
flat       = frcnn.roi_heads.box_head(box_feats)   # triggers hook on box_head[3]
class_logits, _ = frcnn.roi_heads.box_predictor(flat)
score = class_logits[0, tgt_lbl]

frcnn.zero_grad(set_to_none=True)
score.backward()
h.remove()

A    = act_store['feat'].detach()       # [1, 256, 7, 7]
grad = act_store['feat'].grad.detach()  # [1, 256, 7, 7]

# Grad-CAM++ weights on the 7×7 RoI grid
g2 = grad ** 2
g3 = g2 * grad
denom = 2 * g2 + (A * g3).sum(dim=(2, 3), keepdim=True)
alpha = g2 / torch.where(denom != 0, denom, torch.ones_like(denom))
weights = (alpha * torch.relu(grad)).sum(dim=(2, 3), keepdim=True)
cam_roi = torch.relu((weights * A).sum(dim=1))[0].cpu().numpy()   # [7, 7]
cam_roi -= cam_roi.min(); cam_roi /= max(cam_roi.max(), 1e-9)

# 4) paste 7×7 CAM back onto an image-shaped canvas, restricted to the box
x1, y1, x2, y2 = map(int, np.round(tgt_box))
x1 = max(0, x1); y1 = max(0, y1)
x2 = min(W_IMG, x2); y2 = min(H_IMG, y2)
canvas = np.zeros((H_IMG, W_IMG), dtype=np.float32)
if x2 > x1 and y2 > y1:
    canvas[y1:y2, x1:x2] = cv2.resize(cam_roi, (x2 - x1, y2 - y1),
                                      interpolation=cv2.INTER_CUBIC)

# Green target box + "frcnn cls=<id> s=<score>" label, matching the other plots.
# tgt_lbl is 1-indexed (0 = background); show tgt_lbl-1 so the class id matches
# the 0-indexed ids displayed in the D-RISE / EigenCAM plots.
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(img_np); ax[0].set_title('Input'); ax[0].axis('off')
draw_box(ax[0], tgt_box, label=f'frcnn cls={tgt_lbl - 1} s={all_scr[idx]:.2f}')
ax[1].imshow(overlay_heatmap(img_np, canvas))
ax[1].set_title('Grad-CAM++ — Faster R-CNN (RoI-head, box_head[3])')
draw_box(ax[1], tgt_box); ax[1].axis('off')
plt.tight_layout()
save_fig(fig, 'xai_gradcampp_fasterrcnn', layer='box_head3', conf=CONF_FASTERRCNN)
plt.show()

## 3 — EigenCAM (YOLO, layer 15, box-masked)

EigenCAM (Muhammad & Yeasin, 2020) is gradient-free: hook a chosen feature map
$A \in \mathbb{R}^{C \times HW}$, take the first principal component, project
the activations onto it, and reshape back to $H \times W$.

Two adjustments for LARS:

1. **Layer 15.** Layer 15 is the stride-8 branch of the YOLOv8
   neck (`128×128` at imgsz=1024), giving ~16× the spatial resolution of
   layer 21 — necessary for the small buoys and distant boats in LARS.
2. **Box-masked SVD.** Before computing the SVD we zero out feature-grid cells
   outside the projected target box, so the first principal component is
   dominated by activations responsible for *that* detection instead of the
   whole scene.

In [ ]:
# ── EigenCAM for YOLOv8 (layer 15, box-masked) ──────────────────────────────
# Two changes vs. the textbook EigenCAM:
#   1) Hook the small-object branch (layer 15 = stride 8) instead of layer 21
#      (stride 32). Many LARS objects span only a few pixels — the stride-32
#      grid is too coarse for them.
#   2) Mask the activation grid to the *target detection's* projected cells
#      BEFORE running SVD, so the first principal component is dominated by
#      the chosen box rather than by the global scene statistics.
#
# Note: pick a target by index in `target_idx` if you want to inspect a
# different detection.

yolo_model = yolo.model.to(DEVICE).eval()

TARGET_LAYER = '21'   # stride-8 feature map → ~highest spatial resolution
target_idx   = 0      # which detection to explain (sorted by confidence)

act_store = {}
def _hook(module, inp, out):
    act_store['feat'] = out.detach()
h = dict(yolo_model.model.named_modules())[TARGET_LAYER].register_forward_hook(_hook)

res = yolo.predict(source=pil_img, imgsz=1024, conf=CONF_YOLO,
                   device=DEVICE, verbose=False)[0]
h.remove()

if len(res.boxes) == 0:
    raise RuntimeError('YOLO produced no detections at the current threshold')

# Sort detections by confidence so target_idx=0 → top detection
order = res.boxes.conf.cpu().numpy().argsort()[::-1]
boxes_sorted = res.boxes.xyxy.cpu().numpy()[order]
scores_sorted = res.boxes.conf.cpu().numpy()[order]
labels_sorted = res.boxes.cls.cpu().numpy().astype(int)[order]
tgt_box   = boxes_sorted[target_idx]
tgt_score = scores_sorted[target_idx]
tgt_cls   = labels_sorted[target_idx]
print(f'Target #{target_idx}: cls={tgt_cls} score={tgt_score:.3f} box={tgt_box.round(1)}')

feat = act_store['feat']                      # [B, C, Hf, Wf]
B, C, Hf, Wf = feat.shape
A_full = feat[0].cpu().numpy()                # [C, Hf, Wf]

# Project target box from image coords → feature grid
sx, sy = Wf / W_IMG, Hf / H_IMG
fx1 = int(np.floor(tgt_box[0] * sx)); fy1 = int(np.floor(tgt_box[1] * sy))
fx2 = int(np.ceil (tgt_box[2] * sx)); fy2 = int(np.ceil (tgt_box[3] * sy))
fx1 = max(0, fx1); fy1 = max(0, fy1)
fx2 = min(Wf, fx2); fy2 = min(Hf, fy2)
print(f'Feature grid {Hf}x{Wf}; box cells x[{fx1}:{fx2}] y[{fy1}:{fy2}] '
      f'({max(fx2-fx1,0)}x{max(fy2-fy1,0)} cells)')

# Build a soft mask (1 inside the projected box, 0 outside) and apply
mask = np.zeros((Hf, Wf), dtype=np.float32)
if fx2 > fx1 and fy2 > fy1:
    mask[fy1:fy2, fx1:fx2] = 1.0
A_masked = A_full * mask[None, :, :]          # zero out unrelated cells

# SVD over the C × (Hf*Wf) matrix of the masked activations
A = A_masked.reshape(C, Hf * Wf)
A = A - A.mean(axis=1, keepdims=True)
U, S, Vt = np.linalg.svd(A, full_matrices=False)
cam = Vt[0].reshape(Hf, Wf)
if cam[fy1:fy2, fx1:fx2].mean() < 0:          # flip sign so high = salient
    cam = -cam
cam *= mask                                    # ignore anything outside the box
cam = cam - cam.min(); cam = cam / max(cam.max(), 1e-9)
cam_big = cv2.resize(cam, (W_IMG, H_IMG), interpolation=cv2.INTER_LINEAR)

# Conform with the other plots: show only the explained target box in green
# (default colour) with a "yolo cls=<id> s=<score>" label, Input + heatmap panels.
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(img_np); ax[0].set_title('Input'); ax[0].axis('off')
draw_box(ax[0], tgt_box, label=f'yolo cls={tgt_cls} s={tgt_score:.2f}')
ax[1].imshow(overlay_heatmap(img_np, cam_big))
ax[1].set_title(f'EigenCAM — YOLO (layer {TARGET_LAYER}, '
                f'box-masked, feat {Hf}x{Wf})')
draw_box(ax[1], tgt_box); ax[1].axis('off')
plt.tight_layout()
save_fig(fig, 'xai_eigencam_yolo',
         layer=TARGET_LAYER, target_idx=target_idx, conf=CONF_YOLO)
plt.show()

## 4 — Cross-attention map (RF-DETR)

RF-DETR's decoder uses **multi-scale deformable attention** (MSDeformAttn): each
object query learns a small set of *sampling locations* in the encoder feature
maps, each with an *attention weight*. Unlike vanilla DETR there is no dense
[num_queries × HW] attention matrix — only sparse sampled points.

We hook the **last decoder layer's `cross_attn`** module, capture its
`sampling_locations` and `attention_weights`, then *scatter* those weights
onto an image-shaped accumulator (separately for the query that produced the
top-confidence detection).

In [ ]:
# ── Cross-attention map for RF-DETR ─────────────────────────────────────────
import torch.nn.functional as F

core   = rfdetr.model.model          # LWDETR nn.Module
decoder = core.transformer.decoder
last_layer = decoder.layers[-1]
ca = last_layer.cross_attn           # MSDeformAttn

captured = {}
def _ca_pre(module, args, kwargs):
    # Re-compute sampling_locations + attention_weights from the inputs,
    # mirroring MSDeformAttn.forward.
    # forward(query, reference_points, input_flatten, input_spatial_shapes,
    #         input_level_start_index, input_padding_mask=None, ...)
    query = args[0]
    reference_points = args[1]
    input_spatial_shapes = args[3]
    N, Len_q, _ = query.shape
    so = module.sampling_offsets(query).view(
        N, Len_q, module.n_heads, module.n_levels, module.n_points, 2)
    aw = module.attention_weights(query).view(
        N, Len_q, module.n_heads, module.n_levels * module.n_points)
    aw = F.softmax(aw, -1).view(N, Len_q, module.n_heads,
                                module.n_levels, module.n_points)
    if reference_points.shape[-1] == 2:
        off_norm = torch.stack(
            [input_spatial_shapes[..., 1], input_spatial_shapes[..., 0]], -1)
        sl = (reference_points[:, :, None, :, None, :]
              + so / off_norm[None, None, None, :, None, :])
    else:
        sl = (reference_points[:, :, None, :, None, :2]
              + so / module.n_points
              * reference_points[:, :, None, :, None, 2:] * 0.5)
    captured['sampling_locations'] = sl.detach()      # [N, Lq, heads, lvls, pts, 2]
    captured['attention_weights']  = aw.detach()      # [N, Lq, heads, lvls, pts]
    captured['spatial_shapes']     = input_spatial_shapes.detach()

h = ca.register_forward_pre_hook(_ca_pre, with_kwargs=True)
dets = rfdetr.predict(pil_img, threshold=CONF_RFDETR)
h.remove()

print(f'RF-DETR: {len(dets)} detections')
sl = captured['sampling_locations'][0]    # [Lq, heads, lvls, pts, 2]
aw = captured['attention_weights'][0]     # [Lq, heads, lvls, pts]
ss = captured['spatial_shapes']           # [lvls, 2]
Lq, heads, lvls, pts, _ = sl.shape
print(f'queries={Lq}  heads={heads}  levels={lvls}  points={pts}  '
      f'level shapes={ss.tolist()}')

# Gaussian blur sigma for the scattered attention maps (encoded in filename too)
SIGMA = 15.0

def scatter_map(query_idx, H=H_IMG, W=W_IMG, sigma=4.0):
    """Sum attention over heads & levels and scatter onto a HxW grid."""
    canvas = torch.zeros(H, W, device=sl.device)
    locs   = sl[query_idx]     # [heads, lvls, pts, 2] in [0,1]
    wts    = aw[query_idx]     # [heads, lvls, pts]
    xs = (locs[..., 0] * W).clamp(0, W - 1).long().reshape(-1)
    ys = (locs[..., 1] * H).clamp(0, H - 1).long().reshape(-1)
    ws = wts.reshape(-1)
    canvas.index_put_((ys, xs), ws, accumulate=True)
    cam = canvas.cpu().numpy()
    k = max(int(2 * sigma * 3) | 1, 3)
    cam = cv2.GaussianBlur(cam, (k, k), sigma)
    cam -= cam.min(); cam /= max(cam.max(), 1e-9)
    return cam

# RFDETRBase.predict returns detections sorted by score; align with query index.
# The .data attribute exposes the underlying query indices when available; if
# not, fall back to the top-Lq queries by max attention magnitude.
scores = np.asarray(dets.confidence)
boxes  = np.asarray(dets.xyxy)
labels = np.asarray(dets.class_id)

# 1. Ziel-Bounding-Box auswählen
tgt_box = boxes[0] 

# 2. Box-Koordinaten auf [0, 1] normalisieren
x1, y1, x2, y2 = tgt_box / np.array([W_IMG, H_IMG, W_IMG, H_IMG])

# 3. Boolean-Maske: Welche Sampling Locations liegen innerhalb der Bounding Box?
in_box_x = (sl[..., 0] >= x1) & (sl[..., 0] <= x2)
in_box_y = (sl[..., 1] >= y1) & (sl[..., 1] <= y2)
in_box = in_box_x & in_box_y  # Shape: [Lq, heads, lvls, pts]

# 4. Summiere die Attention-Gewichte (aw) nur für Punkte in der Box
attention_in_box = (aw * in_box.float()).sum(dim=(1, 2, 3)) # Shape: [Lq]

# 5. Top Query (Einzelbild)
top_q = int(attention_in_box.argmax().item())
print(f"Mapped Detection to Query #{top_q}")
cam_top = scatter_map(top_q, sigma=SIGMA)

# 6. Top 20 Queries aggregieren
K = min(20, Lq)
top_qs = np.argsort(-attention_in_box.cpu().numpy())[:K]
cam_agg = np.mean([scatter_map(int(q), sigma=SIGMA) for q in top_qs], axis=0)
cam_agg -= cam_agg.min(); cam_agg /= max(cam_agg.max(), 1e-9)

# --- Plotting ---
fig, ax = plt.subplots(1, 3, figsize=(22, 6))
ax[0].imshow(img_np); ax[0].set_title('Input + detections'); ax[0].axis('off')
for b, s, c in zip(boxes, scores, labels):
    draw_box(ax[0], b, label=f'{c}:{s:.2f}')

ax[1].imshow(overlay_heatmap(img_np, cam_top))
ax[1].set_title(f'X-attn — single query #{top_q}')
ax[1].axis('off')

ax[2].imshow(overlay_heatmap(img_np, cam_agg))
ax[2].set_title(f'X-attn — top {K} queries aggregated')
ax[2].axis('off')

plt.tight_layout()
save_fig(fig, 'xai_xattn_rfdetr', sigma=SIGMA, topk=K, conf=CONF_RFDETR)
plt.show()